![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 17 -- Lab 2: Olympiad Practice Admissions in PyTorch

Olympiad practice, homework. The admissions task of Day 08's Lab 3 as an Olympiad hands it to you: a `train.csv`, a `test.csv` without the target, and one line in the statement: *metric: ROC-AUC*. You read the CSV with pandas, turn the table into NumPy, then into tensors and a TensorDataset, build a network with today's recipe, track the judge's metric on validation rows with early stopping, compare with the logistic regression of Week 2, and write the submission file of probabilities.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** The HAIO 2025 online qualifier *University Admissions*: `train.csv` (6,885 students, 30 columns, yes/no target, 29.8% yes) and `test.csv` (765 students), read from GitHub in the setup cell with the column names translated.

---
# Setup

In [ ]:
# numpy, pandas, matplotlib, scikit-learn and torch are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q numpy pandas matplotlib scikit-learn torch

In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
torch.manual_seed(0)

In [ ]:
BASE = "https://raw.githubusercontent.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad/main/2025/nyari-online/adatok/"
EN = {"Életkor": "age", "Nem": "sex", "Osztályzat_9": "grade_9", "Osztályzat_10": "grade_10", "Osztályzat_11": "grade_11",
      "Osztályzat_12": "grade_12", "Történelem": "history", "Matematika": "math", "Magyar Nyelv és Irodalom": "hungarian",
      "Informatika": "informatics", "Biológia": "biology", "Fizika": "physics", "Angol": "english", "Német": "german",
      "Informatika_emelt": "informatics_adv", "Biológia_emelt": "biology_adv", "Fizika_emelt": "physics_adv",
      "Angol_emelt": "english_adv", "Német_emelt": "german_adv", "Matematika_emelt": "math_adv", "Történelem_emelt": "history_adv",
      "Magyar Nyelv és Irodalom_emelt": "hungarian_adv", "Szülői Végzettség": "parent_education",
      "Középiskola Presztízse": "school_prestige", "Extrakurrikuláris Tevékenységek": "extracurricular",
      "Tanulási Szokások": "study_habits", "Munkatapasztalat": "work_experience", "Ajánlások Száma": "recommendations",
      "Versenyeken Való Részvétel": "competitions", "Vármegye": "county", "Felvételi Eredmény": "admitted", "ID": "ID"}

In [ ]:
def prepare(d):
    """Day 08 Lab 3 in one function: -1 becomes a flag plus a zero, the county becomes one-hot columns."""
    d = d.copy()
    for c in ["informatics", "biology", "physics", "english", "german"]:
        d[c + "_taken"] = (d[c] != -1).astype(int)
        d[c] = d[c].replace(-1, 0)
        d[c + "_adv"] = d[c + "_adv"].replace(-1, 0)
    feats = [c for c in d.columns if c not in ("county", "admitted", "ID")]
    return pd.concat([d[feats], pd.get_dummies(d["county"], prefix="county", dtype=int)], axis=1)

---
# Part 1 -- From CSV to tensors

The path every tabular Olympiad task takes: pandas for the file and the cleaning, NumPy for the numbers, tensors and a TensorDataset for the loop.

## Task 1: Read the CSV files

Read `BASE + "train.csv"` and `BASE + "test.csv"` with `pd.read_csv` and rename the columns with `.rename(columns=EN)` into `train` and `test`. Print both shapes and the share of admitted students, `train["admitted"].mean()` (expected 0.298).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert train.shape == (6885, 32) and test.shape == (765, 31), 'read_csv on BASE + file name, then rename(columns=EN)'
print('Task 1 passed')

## Task 2: From the table to NumPy

Apply `prepare` to both tables: `F = prepare(train)` and `F_test = prepare(test).reindex(columns=F.columns, fill_value=0)` (the test table must have the same columns in the same order). Then `X_np = F.values.astype(np.float32)`, `y_np = train["admitted"].astype(np.float32).values` (a float target for `BCEWithLogitsLoss`) and `X_test_np = F_test.values.astype(np.float32)`. Print the three shapes (54 columns).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert X_np.shape == (6885, 54) and X_test_np.shape == (765, 54) and y_np.dtype == np.float32, 'prepare both tables, reindex the test columns, cast to float32'
print('Task 2 passed')

## Task 3: Split and standardise

Split with `train_test_split(X_np, y_np, test_size=0.2, random_state=42, stratify=y_np)` into `X_tr, X_va, y_tr, y_va`. Compute `mu = X_tr.mean(axis=0)` and `sd = X_tr.std(axis=0)`, set `sd[sd < 1e-6] = 1.0`, and standardise the three feature arrays with `mu, sd` from the training rows (`X_tr_s, X_va_s, X_test_s`). Print the shapes and the share of admitted in `y_va`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert X_tr_s.shape == (5508, 54) and abs(X_tr_s.mean()) < 0.01, 'standardise with the training mean and spread; constant columns keep sd = 1'
print('Task 3 passed')

## Task 4: Tensors, a TensorDataset, a DataLoader

Turn the standardised arrays into `float32` tensors `Xt, Xv, Xte` and the labels into `float32` tensors `yt, yv`. Build `train_ds = TensorDataset(Xt, yt)` and `train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)`. Print `len(train_loader)` (87) and the first batch's shapes.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_loader) == 87 and yt.dtype == torch.float32, 'TensorDataset(Xt, yt) with float32 labels, batches of 64'
print('Task 4 passed')

---
# Part 2 -- The line to beat, and the plain network

## Task 5: Logistic regression on the same split

Fit `make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))` on `X_tr, y_tr` and compute the validation ROC-AUC of `predict_proba(X_va)[:, 1]` as `auc_logistic`. Expected about 0.956 (Day 10 found 0.954 on a different split). This is the number the network has to match.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert auc_logistic > 0.94, 'logistic: scaled LogisticRegression on X_tr, ROC-AUC of predict_proba[:, 1] on X_va'
print('Task 5 passed')

## Task 6: Evaluate by ROC-AUC

Write `evaluate(model, X, y)` returning `(loss, auc)` for a whole tensor `X` and labels `y`: eval mode, `no_grad`, `logits = model(X).squeeze(1)`, loss from `loss_fn = nn.BCEWithLogitsLoss()`, probabilities `torch.sigmoid(logits)`, and `roc_auc_score(y.numpy(), probs.numpy())`. ROC-AUC wants probabilities, not labels (Day 09).

In [ ]:
# Your code here


## Task 7: The loop with early stopping on ROC-AUC

Write `fit(model, opt, epochs, patience=None)`: per epoch the four moves over `train_loader` with `loss_fn(model(xb).squeeze(1), yb)`, then `evaluate` on `(Xt, yt)` and `(Xv, yv)`; `history` holds `(train_loss, val_loss, train_auc, val_auc)`; keep `best_state` by validation AUC and stop after `patience` epochs without improvement; return `history, (best_auc, best_epoch, best_state)`. Train the plain network `nn.Sequential(nn.Linear(54, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))` with `SGD(lr=0.05)` for 40 epochs (no patience) as `hist_plain, best_plain`. Print the best validation AUC and its epoch, and the final training and validation AUC. Expected: best about 0.95 around epoch 20, then the validation AUC falls towards 0.92 by epoch 40 while the training AUC keeps rising.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(hist_plain) == 40 and best_plain[0] > 0.9, 'fit: 40 epochs of SGD, best state by validation ROC-AUC'
print('Task 7 passed')

---
# Part 3 -- The recipe, the comparison, the submission

## Task 8: The recipe

Write `make_recipe()`: `Linear(54, 128), BatchNorm1d(128), ReLU, Dropout(0.3), Linear(128, 64), BatchNorm1d(64), ReLU, Dropout(0.3), Linear(64, 1)`. Train it with `AdamW(lr=1e-3, weight_decay=0.01)`, up to 60 epochs, `patience=8`, as `hist_recipe, best_recipe`. Print the best validation AUC, its epoch, the epoch the run stopped at, and compare with `auc_logistic`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert best_recipe[0] > best_plain[0] - 0.005 and len(hist_recipe) < 60, 'recipe: BatchNorm and dropout, AdamW, early stopping with patience 8'
print('Task 8 passed')

## Task 9: Plain against recipe

Left: training and validation AUC per epoch for both runs (plain solid, recipe dashed) with `auc_logistic` as a horizontal line. Right: training and validation losses of both. Which run overfits, and when?

In [ ]:
# Your code here


## Task 10: The submission

Load `best_recipe[2]` into a fresh `make_recipe()`, compute the probabilities for `Xte` (eval mode, `no_grad`, `torch.sigmoid(model(Xte).squeeze(1))`), and save `pd.DataFrame({"ID": test["ID"], "admitted": probabilities})` to `submission.csv` with `index=False`. Read it back and print its shape and the first rows: 765 rows, numbers between 0 and 1, not labels (Day 09: ROC-AUC needs probabilities).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert check.shape == (765, 2) and check['admitted'].between(0, 1).all() and check['admitted'].nunique() > 10, 'submission.csv: ID and admitted (probabilities), 765 rows'
print('Task 10 passed')

---
## Task 11: Reflect

The recipe beat the plain network and stopped itself, yet logistic regression stayed within a few thousandths of it. Which model would you submit, and what does Day 10's *No Free Lunch* slide say about this table? The plain run's best epoch was found on the validation rows and so was the recipe's stopping point. What does that do to the validation AUC as an estimate of the hidden test score, and what from Day 13 would fix it? List the five steps that took the CSV to a TensorDataset. Which one would you write first in a two-hour competition, and which one is the easiest to get subtly wrong?

*Your answers here*

---
## Conclusion

- **CSV to tensors**: pandas read and cleaned the table, NumPy held the numbers, the training rows gave the mean and spread, and `TensorDataset` plus a `DataLoader` fed the loop.
- **The judge's metric inside the loop**: ROC-AUC on probabilities, computed every epoch, chose the checkpoint and the stopping point.
- **The recipe**: BatchNorm, dropout, AdamW and early stopping turned a network that overfit after epoch 15 into one that stays flat and stops itself.
- **The comparison habit**: the line from Week 2 was the baseline, the network had to earn its place, and the submission carried probabilities, not labels.

---

Made By **Sattam Altwaim**